# Digit Classification

In [2]:
import torch
import torch.nn as nn
import torchvision
import torch.optim as optim
from torchvision import datasets,transforms

In [6]:
transform =transforms.ToTensor()
train_dataset=datasets.MNIST(root="./data",train=True,download=True,transform=transform)
test_dataset=datasets.MNIST(root="./data",train=False,download=True,transform=transform)

In [7]:
from torch.utils.data import DataLoader
train_loader=DataLoader(train_dataset,batch_size=64,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=64)

# Recurrent Neural Network

In [9]:
class RNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn=nn.RNN(
            input_size=28,
            hidden_size=128,
            num_layers=2,
            batch_first=True
        )
        self.fc=nn.Linear(128,10)
    def forward(self,x):
        output,hidden=self.rnn(x)
        hidden=hidden[-1]
        out=self.fc(hidden)
        return out   

In [10]:
model1=RNN()

In [11]:
criterion=nn.CrossEntropyLoss()
optimizer1=optim.Adam(model1.parameters())

In [17]:
epochs=10
for epoch in range(epochs):
    model1.train()
    running_loss=0.0
    for images,labels in train_loader:
        images=images.squeeze(1)
        optimizer1.zero_grad()
        outputs=model1(images)
        loss=criterion(outputs,labels)
        loss.backward()
        optimizer1.step()
        running_loss+=loss.item()
    print(f"Epoch {epoch+1}/{epochs} Loss : {running_loss/len(train_loader):.4f}")     

Epoch 1/10 Loss : 0.1289
Epoch 2/10 Loss : 0.1247
Epoch 3/10 Loss : 0.1167
Epoch 4/10 Loss : 0.1137
Epoch 5/10 Loss : 0.1097
Epoch 6/10 Loss : 0.1235
Epoch 7/10 Loss : 0.1117
Epoch 8/10 Loss : 0.1037
Epoch 9/10 Loss : 0.0963
Epoch 10/10 Loss : 0.0930


In [19]:
model1.eval()
correct=0
total=0
with torch.no_grad():
    for images,labels in test_loader:
        images=images.squeeze(1)
        outputs=model1(images)
        _,predicted=torch.max(outputs,dim=1)
        total+=labels.size(0)
        correct+=(predicted==labels).sum().item()
accuracy=correct/total
print(f"Accuracy : {accuracy:.4f}")

Accuracy : 0.9668


# Convolution Neural Network

In [34]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features=nn.Sequential(
            nn.Conv2d(
                in_channels=1,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),
            
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier=nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7,128),
            nn.ReLU(),
            nn.Linear(128,10)
        )
    def forward(self,x):
        x=self.features(x)
        x=self.classifier(x)
        return x

In [35]:
model2=CNN()
optimizer2=optim.Adam(model2.parameters())

In [41]:
epochs=10
for epoch in range(epochs):
    model2.train()
    running_loss=0.0
    for images,labels in train_loader:
        optimizer2.zero_grad()
        outputs=model2(images)
        loss=criterion(outputs,labels)
        loss.backward()
        optimizer2.step()
        running_loss+=loss.item()
    print(f"Epoch {epoch+1}/{epochs}: {running_loss/len(train_loader):.4f}")     

Epoch 1/10: 0.0149
Epoch 2/10: 0.0117
Epoch 3/10: 0.0075
Epoch 4/10: 0.0074
Epoch 5/10: 0.0067
Epoch 6/10: 0.0052
Epoch 7/10: 0.0065
Epoch 8/10: 0.0034
Epoch 9/10: 0.0050
Epoch 10/10: 0.0033


In [47]:
model2.eval()
correct=0
total=0
with torch.no_grad():
    for images,labels in test_loader:
        outputs=model2(images)
        _,predicted=torch.max(outputs,1)
        total+=labels.size(0)
        correct+=(labels==predicted).sum().item()
print(f"Accuracy : {correct/total:.4f}")

Accuracy : 0.9922


# LSTM RNN

In [51]:
class LSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm=nn.LSTM(
            input_size=28,
            hidden_size=128,
            num_layers=2,
            batch_first=True
        )
        self.fc=nn.Linear(128,10)
    def forward(self,x):
        output,(hidden,cell)=self.lstm(x)
        hidden=hidden[-1]
        out=self.fc(hidden)
        return out

In [52]:
model3=LSTM()
optimizer3=optim.Adam(model3.parameters())

In [54]:
epochs=10
for epoch in range(epochs):
    model3.train()
    running_loss=0.0
    for images,labels in train_loader:
        images=images.squeeze(1)
        optimizer3.zero_grad()
        outputs=model3(images)
        loss=criterion(outputs,labels)
        loss.backward()
        optimizer3.step()
        running_loss+=loss.item()
    print(f"Epoch {epoch+1}/{epochs}: {running_loss/len(train_loader):.4f}")    

Epoch 1/10: 0.0389
Epoch 2/10: 0.0353
Epoch 3/10: 0.0290
Epoch 4/10: 0.0257
Epoch 5/10: 0.0238
Epoch 6/10: 0.0204
Epoch 7/10: 0.0202
Epoch 8/10: 0.0160
Epoch 9/10: 0.0141
Epoch 10/10: 0.0153


In [56]:
model3.eval()
correct=0
total=0
with torch.no_grad():
    for images,labels in test_loader:
        images=images.squeeze(1)
        outputs=model3(images)
        _,predicted = torch.max(outputs,1)
        total += labels.size(0)
        correct += (predicted==labels).sum().item()

print(f"Accuracy : {correct/total:.4f}")

Accuracy : 0.9889


In [57]:
torch.save(model1.state_dict(), "rnn_model.pth")
torch.save(model3.state_dict(), "lstm_model.pth")
torch.save(model2.state_dict(), "cnn_model.pth")